In [1]:
import torch
from torch import nn

In [ ]:
class vaemodel1(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.mu = nn.Linear(256, 6)  # Reduced latent space size
        self.std = nn.Linear(256, 6)  # Reduced latent space size

    def forward(self, x):
        a = self.encoder(x).permute(0,2,3,1)
        a = torch.flatten(a, start_dim=1)
        mu = self.mu(a)
        lvar = self.std(a)
        out = torch.cat((mu, lvar), dim=1)
        return out

model = vaemodel1()

In [ ]:
model.load_state_dict(torch.load("encoder_pre_trained_w.pt", weights_only=True))

<All keys matched successfully>

In [ ]:
tensor_x = torch.rand((1, 3, 128, 256), dtype=torch.float32)
torch.onnx.export(
    model,
    (tensor_x,),
    f"{model.__class__.__name__}.onnx",  # filename of the ONNX model
    input_names=["input"],  # Rename inputs for the ONNX model
    dynamo=False,  # True or False to select the exporter to use
)

/var/folders/l4/2t40bmn574dd8r6qvmgyvjbh0000gn/T/ipykernel_62987/2956023606.py:33: UserWarning: All inputs of this cat operator must share the same quantization parameters. Otherwise large numerical inaccuracies may occur. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/quantized/cpu/TensorShape.cpp:168.)
  out = torch.cat((mu, lvar), dim=1)
/Users/pantunes/Documents/PhD/Research/ONNX2FPGA/.venv/lib/python3.9/site-packages/torch/jit/_trace.py:475: UserWarning: All inputs of this cat operator must share the same quantization parameters. Otherwise large numerical inaccuracies may occur. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/quantized/cpu/TensorShape.cpp:168.)
  outs = wrap_retval(mod(*_clone_inputs(inputs)))


Torch IR graph at exception: graph(%x : Float(1, 3, 128, 256, strides=[98304, 32768, 256, 1], requires_grad=0, device=cpu),
      %encoder.0._packed_params : __torch__.torch.classes.quantized.Conv2dPackedParamsBase,
      %encoder.1.running_var : Float(16, strides=[1], requires_grad=0, device=cpu),
      %encoder.1.running_mean : Float(16, strides=[1], requires_grad=0, device=cpu),
      %encoder.1.bias : Float(16, strides=[1], requires_grad=0, device=cpu),
      %encoder.1.weight : Float(16, strides=[1], requires_grad=0, device=cpu),
      %encoder.3._packed_params : __torch__.torch.classes.quantized.Conv2dPackedParamsBase,
      %encoder.4.running_var : Float(32, strides=[1], requires_grad=0, device=cpu),
      %encoder.4.running_mean : Float(32, strides=[1], requires_grad=0, device=cpu),
      %encoder.4.bias : Float(32, strides=[1], requires_grad=0, device=cpu),
      %encoder.4.weight : Float(32, strides=[1], requires_grad=0, device=cpu),
      %encoder.6._packed_params : __torch_

UnsupportedOperatorError: Exporting the operator 'quantized::batch_norm2d' to ONNX opset version 17 is not supported. Please feel free to request support or submit a pull request on PyTorch GitHub: https://github.com/pytorch/pytorch/issues.